In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from folium.plugins import TimestampedGeoJson, Timeline, TimelineSlider
from folium.features import GeoJson
from folium.utilities import JsCode
import folium
import plotly.express as px
from itables import show
import itables
import json

itables.init_notebook_mode()

In [ ]:
df_raw = pd.read_csv(
    "data/raw/ratsastie20250813_0706.csv", 
    names=["topic", "payload", "zero", "boolean", "timestamp", "time_delta_prev"],
    header=None,
)
show(df_raw)

In [ ]:
df_raw.info()

In [ ]:
def parse_tracking_payload(payload: str) -> Point:
    try:
        data = json.loads(payload)
    except (TypeError, json.JSONDecodeError):
        return []
    return data.get("objects", [])
    

In [ ]:
def parse_raw_gdf(raw_data: pd.DataFrame) -> pd.DataFrame:
    raw_data = raw_data.dropna(subset=["topic"])
    tracking_mask = raw_data["topic"].str.endswith("tracking")
    tracking_df = raw_data[tracking_mask].copy()

    tracking_df["parsed_payload"] = tracking_df["payload"].apply(parse_tracking_payload)
    tracking_df = tracking_df[tracking_df["parsed_payload"].map(len) > 0]

    tracking_df = tracking_df.explode("parsed_payload", ignore_index=True)
    payload_df = pd.json_normalize(tracking_df["parsed_payload"])

    final_df = pd.concat([
        tracking_df[["timestamp", "time_delta_prev"]].reset_index(drop=True),
        payload_df,
    ], axis=1)

    final_df = final_df.drop(columns=final_df.filter(like="bounding_box").columns)
    final_df = final_df.drop(columns=final_df.filter(like="collision").columns)
    final_df = final_df.drop(columns=final_df.filter(like="acceleration").columns)
    final_df = final_df.drop(columns=final_df.filter(like="mass").columns)
    final_df = final_df.drop(columns=final_df.filter(like="previous").columns)
    final_df = final_df.drop(columns=final_df.filter(like="is_active").columns)
    final_df.rename(lambda x: x.replace(".", "_"), axis=1, inplace=True)
    final_df.sort_values(by="timestamp", inplace=True)
    final_df["time_index"] = final_df.groupby("timestamp").ngroup()

    return final_df


parsed_df = parse_raw_gdf(df_raw)

In [ ]:
show(parsed_df)

In [ ]:
parsed_df.timestamp.diff().mean()

In [ ]:
parsed_df.info()

In [ ]:
import numpy as np
import pandas as pd

class Track:
    def __init__(self, x0, t0, row_idx, track_id):
        """
        x0: initial position (x, y)
        t0: initial time (float)
        row_idx: index in the sorted dataframe for this detection
        track_id: internal sequential ID
        """
        self.id = track_id
        
        # State: [x, y, vx, vy]
        self.x = np.array([x0[0], x0[1], 0.0, 0.0], dtype=float)
        self.P = np.eye(4) * 10.0  # large initial uncertainty
        
        self.first_time = t0
        self.last_time = t0
        
        self.hits = 1
        self.confirmed = False
        
        # Keep which rows in the df belong to this track
        self.row_indices = [row_idx]

    def predict(self, t_now, process_noise_pos=1.0, process_noise_vel=1.0):
        dt = t_now - self.last_time
        if dt <= 0:
            return  # nothing to do

        # State transition
        F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0 ],
            [0, 0, 0, 1 ],
        ], dtype=float)

        # Process noise
        q_pos = process_noise_pos
        q_vel = process_noise_vel
        Q = np.diag([q_pos, q_pos, q_vel, q_vel])

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + Q

    def update(self, z, t_now, row_idx, meas_noise_pos=0.5):
        # Measurement matrix: we observe position only
        H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], dtype=float)

        R = np.eye(2) * meas_noise_pos

        z = np.asarray(z, dtype=float)
        y = z - H @ self.x                      # innovation
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)     # Kalman gain

        self.x = self.x + K @ y
        I = np.eye(4)
        self.P = (I - K @ H) @ self.P

        self.last_time = t_now
        self.hits += 1
        self.row_indices.append(row_idx)


def build_tracklets_from_detections(
    df,
    v_max=10.0,                  # m/s, ~36 km/h
    max_gap_seconds=0.3,         # we don't allow gaps larger than this (but we kill earlier anyway)
    confirm_min_duration=0.5,    # track must live at least this long to be "confirmed"
    min_tracklet_duration=0.5    # keep only tracklets at least this long
):
    """
    Build conservative tracklets from detections in df.
    df must have columns: 'timestamp', 'centroid_x', 'centroid_y'
    
    Returns a copy of df with a new column 'tracklet_id' (int, -1 for unassigned).
    """

    # Work on a sorted copy, but remember original indices
    df_sorted = df.sort_values("timestamp").reset_index(drop=False)
    original_index = df_sorted["index"].to_numpy()
    
    n = len(df_sorted)
    tracklet_ids_sorted = np.full(n, fill_value=-1, dtype=int)

    # Group detections by timestamp
    # Each group key is timestamp, value is array of row indices in df_sorted
    grouped = df_sorted.groupby("timestamp").indices
    sorted_times = np.array(sorted(grouped.keys()))

    active_tracks = []
    next_internal_id = 0      # internal track counter
    next_tracklet_id = 0      # external tracklet ID we write to df

    def finalize_track(track):
        nonlocal next_tracklet_id
        
        duration = track.last_time - track.first_time
        if (not track.confirmed) or (duration < min_tracklet_duration):
            # discard weak/short tracks
            return
        
        # Assign a new global tracklet_id to all rows in this track
        tracklet_ids_sorted[track.row_indices] = next_tracklet_id
        next_tracklet_id += 1

    for t in sorted_times:
        row_idxs = grouped[t]
        det_x = df_sorted.loc[row_idxs, "centroid_x"].to_numpy()
        det_y = df_sorted.loc[row_idxs, "centroid_y"].to_numpy()
        detections = np.stack([det_x, det_y], axis=1)  # shape (M, 2)

        # 1) Predict all active tracks forward to time t,
        #    and drop tracks that have grown too old.
        still_active = []
        for track in active_tracks:
            dt = t - track.last_time
            if dt > max_gap_seconds:
                # end of tracklet
                finalize_track(track)
            else:
                track.predict(t)
                still_active.append(track)
        active_tracks = still_active

        # 2) Associate tracks with detections (greedy nearest-neighbor with gating)
        num_tracks = len(active_tracks)
        num_dets = len(detections)

        unmatched_track_indices = set(range(num_tracks))
        unmatched_det_indices = set(range(num_dets))

        matches = []

        if num_tracks > 0 and num_dets > 0:
            # Build distance matrix between predicted track positions and detections
            track_positions = np.array([[tr.x[0], tr.x[1]] for tr in active_tracks])
            # (T, D) pairwise distances
            diff = track_positions[:, None, :] - detections[None, :, :]
            dists = np.linalg.norm(diff, axis=2)

            # For each track, compute the max allowed distance based on v_max and dt
            dt_array = np.array([t - tr.last_time for tr in active_tracks])
            # avoid dt <= 0 just in case
            dt_array = np.maximum(dt_array, 1e-6)
            max_dist_for_track = v_max * dt_array   # shape (T,)

            # Gating mask: valid if d <= v_max * dt
            gate = dists <= max_dist_for_track[:, None]

            # Set invalid pairs to a very large distance
            LARGE = 1e9
            dists_masked = dists.copy()
            dists_masked[~gate] = LARGE

            # Greedy: pick smallest distance, then remove that track and detection, repeat
            while True:
                min_idx = np.argmin(dists_masked)
                i, j = divmod(min_idx, num_dets)
                if dists_masked[i, j] >= LARGE:
                    break  # no more valid pairs

                # Accept this match
                matches.append((i, j))
                unmatched_track_indices.discard(i)
                unmatched_det_indices.discard(j)

                # Remove this row and column from further consideration
                dists_masked[i, :] = LARGE
                dists_masked[:, j] = LARGE

        # 3) Update matched tracks
        for ti, di in matches:
            track = active_tracks[ti]
            det_idx = row_idxs[di]
            z = detections[di]
            track.update(z, t_now=t, row_idx=det_idx)

            # Check if this track should now be confirmed
            duration = track.last_time - track.first_time
            if (not track.confirmed) and (duration >= confirm_min_duration):
                track.confirmed = True

        # 4) Finalize unmatched tracks (we don't allow them to survive unmatched)
        tracks_to_keep = []
        for idx, track in enumerate(active_tracks):
            if idx in unmatched_track_indices:
                # finalize and discard
                finalize_track(track)
            else:
                tracks_to_keep.append(track)
        active_tracks = tracks_to_keep

        # 5) Create new tentative tracks for unmatched detections
        for di in unmatched_det_indices:
            det_idx = row_idxs[di]
            x0 = detections[di]
            new_track = Track(x0=x0, t0=t, row_idx=det_idx, track_id=next_internal_id)
            next_internal_id += 1
            active_tracks.append(new_track)

    # After last time step, finalize any remaining tracks
    for track in active_tracks:
        finalize_track(track)

    # Map tracklet IDs back to original df order
    out = df.copy()
    out["tracklet_id"] = -1
    out.loc[original_index, "tracklet_id"] = tracklet_ids_sorted

    return out

In [ ]:
tracklet_df = build_tracklets_from_detections(
    parsed_df,
    v_max=10.0,               # ~36 km/h
    max_gap_seconds=0.5,
    confirm_min_duration=0.3,
    min_tracklet_duration=0.3,
)

tracklet_df["tracklet_id"].value_counts().head()

In [ ]:
tracklet_df.info()

In [ ]:
tracklet_df.tracklet_id.unique().size

In [ ]:
SENSOR_LAT = 60.197547
SENSOR_LON = 24.907931

def lidar_xy_to_latlon(x, y, sensor_lat=SENSOR_LAT, sensor_lon=SENSOR_LON, rotation_deg=0.0):
    """
    Convert local LiDAR coordinates (meters) to WGS84 lat/lon using a simple
    flat-earth approximation around the sensor location.
    
    rotation_deg: rotation of LiDAR X axis relative to East (positive CCW).
                  0°  => x=east,  y=north
                  170° => roughly facing south (you can experiment with this)
    """
    # 1) rotate in the local ENU plane
    theta = np.deg2rad(rotation_deg)
    xr = x * np.cos(theta) - y * np.sin(theta)
    yr = x * np.sin(theta) + y * np.cos(theta)

    # 2) convert meters to lat/lon offsets
    R = 6378137.0  # Earth radius [m]
    dlat = (yr / R) * (180.0 / np.pi)
    dlon = (xr / (R * np.cos(np.deg2rad(sensor_lat)))) * (180.0 / np.pi)

    lat = sensor_lat + dlat
    lon = sensor_lon + dlon
    return lat, lon

In [ ]:
def most_common(s: pd.Series):
    mode = s.mode()
    return mode.iloc[0] if len(mode) else None

def make_line(points):
    pts = [(p.x, p.y) for p in points if p.is_valid]
    return LineString(pts) if len(pts) > 1 else None

def build_tracklet_geodata(tracklet_df: pd.DataFrame) -> gpd.GeoDataFrame:
    # Keep only assigned tracklets
    df = tracklet_df[tracklet_df["tracklet_id"] >= 0].copy()
    if df.empty:
        raise ValueError("No tracklets to visualize (tracklet_id < 0 only).")

    # Sort by tracklet + time
    df = df.sort_values(["tracklet_id", "timestamp"]).copy()

    # Convert timestamp to datetime (seconds!)
    df["timestamp_dt"] = pd.to_datetime(df["timestamp"], unit="s")

    # Geometry in LiDAR local coordinates
    df["center"] = df.apply(lambda r: Point(r["centroid_x"], r["centroid_y"]), axis=1)
    gdf_tracks = gpd.GeoDataFrame(df, geometry="center", crs=None)

    # Build one line per tracklet_id
    tracks = (
        gdf_tracks.groupby("tracklet_id")
        .agg(
            start_timestamp=("timestamp","min"),
            end_timestamp=("timestamp","max"),
            start_timestamp_dt=("timestamp_dt","min"),
            end_timestamp_dt=("timestamp_dt","max"),
            point_count=("center","count"),
            predicted_class=("predicted_class", most_common) if "predicted_class" in gdf_tracks.columns else ("tracklet_id", "count"),
            geometry=("center", make_line),
        )
        .reset_index()
        .rename(columns={"tracklet_id": "id"})  # for easier reuse of your patterns
    )

    # Handle degenerate tracks (only one point → no LineString)
    tracks_ok = tracks[tracks.geometry.notna()].copy()
    broken = tracks[tracks.geometry.isna()].copy()

    if not broken.empty:
        broken = broken.merge(
            gdf_tracks[["tracklet_id", "center"]],
            left_on="id",
            right_on="tracklet_id",
            how="left"
        )
        broken["geometry"] = broken["center"]
        broken = broken.drop(columns=["center", "tracklet_id"])

    tracks_ok["track_type"] = "TRACK"
    if not broken.empty:
        broken["track_type"] = "BROKEN_POINT"
        tracks_all = pd.concat([tracks_ok, broken], ignore_index=True)
    else:
        tracks_all = tracks_ok.copy()
        tracks_all["track_type"] = "TRACK"

    tracks_all = gpd.GeoDataFrame(tracks_all, geometry="geometry", crs=None)

    # Simple color palette
    PALETTE = ["red", "blue", "green", "orange", "purple"]
    tracks_all["color"] = [PALETTE[i % len(PALETTE)] for i in range(len(tracks_all))]

    return tracks_all

In [ ]:
def tracks_to_wgs84(tracks_all: gpd.GeoDataFrame, rotation_deg: float = 0.0) -> gpd.GeoDataFrame:
    rows = []
    for _, row in tracks_all.iterrows():
        geom = row.geometry
        if geom is None or not geom.is_valid:
            continue

        if geom.geom_type == "LineString":
            coords = []
            for x, y in geom.coords:
                lat, lon = lidar_xy_to_latlon(x, y, rotation_deg=rotation_deg)
                coords.append((lon, lat))  # GeoJSON / WGS84 order: (lon, lat)
            new_geom = LineString(coords)
        elif geom.geom_type == "Point":
            x, y = geom.x, geom.y
            lat, lon = lidar_xy_to_latlon(x, y, rotation_deg=rotation_deg)
            new_geom = Point(lon, lat)
        else:
            # ignore other geometry types for now
            continue

        new_row = row.copy()
        new_row.geometry = new_geom
        rows.append(new_row)

    gdf_wgs84 = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    return gdf_wgs84

In [ ]:
def build_tracklet_timeline_map(
    tracklet_df: pd.DataFrame,
    output_html: str = "track_visualizations/tracklets_timeline_slider.html",
    rotation_deg: float = 0.0,
):
    tracks_all = build_tracklet_geodata(tracklet_df)
    tracks_all_wgs84 = tracks_to_wgs84(tracks_all, rotation_deg=rotation_deg)

    # Build GeoJSON manually (same pattern as your working code)
    features = []
    for _, row in tracks_all_wgs84.iterrows():
        geom = row.geometry
        if geom is None or not geom.is_valid:
            continue

        if geom.geom_type == "LineString":
            coordinates = [[x, y] for x, y in geom.coords]
        elif geom.geom_type == "Point":
            coordinates = [geom.x, geom.y]
        else:
            continue

        features.append({
            "type": "Feature",
            "geometry": {
                "type": geom.geom_type,
                "coordinates": coordinates,
            },
            "properties": {
                "id": row["id"],
                "track_type": row["track_type"],
                "predicted_class": row.get("predicted_class", None),
                "point_count": int(row["point_count"]),
                "start": float(row["start_timestamp"]),
                "end": float(row["end_timestamp"]),
                "start_timestamp_dt": row["start_timestamp_dt"].isoformat(),
                "end_timestamp_dt": row["end_timestamp_dt"].isoformat(),
                "color": row["color"],
            },
        })

    tracks_geojson = {
        "type": "FeatureCollection",
        "features": features,
    }

    # JS style function same as your pattern
    style_js = JsCode("""
        function (data) {
            return {
                color: data.properties.color,
                weight: 3,
                opacity: 0.8
            };
        }
    """)

    # Center map around average centroid
    centroids = tracks_all_wgs84.geometry.centroid
    center_lat = centroids.y.mean()
    center_lon = centroids.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=18)

    # Reuse your Timeline / TimelineSlider pattern
    timeline = Timeline(
        data=tracks_geojson,
        style=style_js,
    ).add_to(m)

    tooltip = folium.GeoJsonTooltip(
        fields=["id", "track_type", "predicted_class", "point_count", "start_timestamp_dt", "end_timestamp_dt"],
        sticky=True,
    )
    tooltip.add_to(timeline)

    TimelineSlider(
        auto_play=False,
        show_ticks=True,
        enable_keyboard_controls=True,
        playback_duration=30000,
    ).add_timelines(timeline).add_to(m)

    m.save(output_html)
    print("Saved →", output_html)

In [ ]:
# After you have tracklet_df from build_tracklets_from_detections(...)
build_tracklet_timeline_map(
    tracklet_df,
    output_html="track_visualizations/tracklets_timeline_slider.html",
    rotation_deg=170.0,  # or try 170.0 later
)